In [ ]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
from urllib.parse import urlparse, parse_qs
import re
import time
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup
import datetime
import os
from selenium import webdriver
from time import sleep
import pandas as pd
from collections import deque
from urllib.parse import urldefrag, urlparse, parse_qs
from urllib.parse import urljoin, urlsplit, urlunsplit
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'LV CBOL' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)



Running LV CBOL Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------


START_URL = "https://uzraudziba.bank.lv/en/market/alternative-investment-fund-managers/"

regdict={

        regulatorName + ' 1': 'https://uzraudziba.bank.lv/en/market/alternative-investment-fund-managers/',
        regulatorName + ' 2': 'https://uzraudziba.bank.lv/en/market/insurance-companies/',
        regulatorName + ' 3': 'https://uzraudziba.bank.lv/en/market/insurance-intermediaries/',
        regulatorName + ' 4': 'https://uzraudziba.bank.lv/en/market/financial-instruments-market/',
        # regulatorName + ' 5': 'https://uzraudziba.bank.lv/en/market/financial-holdings/',
        # regulatorName + ' 6': 'https://uzraudziba.bank.lv/en/market/investment-service-providers/',
        # regulatorName + ' 7_1': 'https://uzraudziba.bank.lv/en/market/investment-management-companies/',
        # regulatorName + ' 7_2': 'https://uzraudziba.bank.lv/en/market/investment-management-companies/foreign-funds/',
        # regulatorName + ' 8': 'https://uzraudziba.bank.lv/en/market/crowdfunding-service-providers/',
        # regulatorName + ' 9': 'https://uzraudziba.bank.lv/en/market/co-operative-credit-unions/',
        # regulatorName + ' 10': 'https://uzraudziba.bank.lv/en/market/credit-institutions/',
        # regulatorName + ' 11': 'https://uzraudziba.bank.lv/en/market/payment-service-providers/',
        # regulatorName + ' 12': 'https://uzraudziba.bank.lv/en/market/pension-funds/',
        # regulatorName + ' 13': 'https://uzraudziba.bank.lv/en/market/foreign-exchange-trading-companies/',


        }



Typology={

    regulatorName + ' 1': 'Investment service providers',
    regulatorName + ' 2': 'Insurance companies',
    regulatorName + ' 3': 'Insurance Intermediaries',
    regulatorName + ' 4': 'Financial instruments market',
    regulatorName + ' 5': 'Financial holdings',
    regulatorName + ' 6': 'Investment service providers',
    regulatorName + ' 7_1': 'Investment management companies',
    regulatorName + ' 7_2': 'Investment management companies',
    regulatorName + ' 8': 'Crowdfunding service providers',
    regulatorName + ' 9': 'Co-operative Credit Unions',
    regulatorName + ' 10': 'Credit institutions',
    regulatorName + ' 11': 'Payment service providers',
    regulatorName + ' 12': 'Pension Funds',
    regulatorName + ' 13': 'Foreign exchange trading companies',

        }

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

SKIP_SUBSTRINGS = 
        [
            "managers-from-eea",
            "service-providers-from-the-eea",
            "central-securities-depositories-from-the-eea",
            "financial-instruments-market/depositary",
            "securities-offerings-by-member-states",
            "registered-foreign-investment-service-providers",
            "tied-agents-of-eea-investment-service-providers".
            "credit-institutions-in-liquidation",
            "credit-institutions-in-reorganization",
            


            # add more here...
        ]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

In [ ]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict
    
def get_soup(url, tries=3, sleep=0.5):
    last = None
    for i in range(tries):
        try:
            r = session.get(url, timeout=30,verify=False)
            r.raise_for_status()
            return BeautifulSoup(r.text, "lxml")
        except Exception as e:
            last = e
            time.sleep(sleep * (i + 1))
    raise last

def norm_label(s):
    return re.sub(r"\s+", " ", (s or "").strip()).rstrip(":").strip().lower()

def extract_post_urls(soup, base_url):
    post_urls = []
    for a in soup.select("div.posts-block a[href]"):
        href = a.get("href")
        if href:
            post_urls.append(urljoin(base_url, href))
    return list(dict.fromkeys(post_urls))

def extract_l_pages(soup, base_url):
    l_vals = set()
    for a in soup.select("a[href*='?l=']"):
        href = a.get("href")
        if not href:
            continue
        abs_href = urljoin(base_url, href)
        qs = parse_qs(urlparse(abs_href).query)
        if "l" in qs:
            try:
                l_vals.add(int(qs["l"][0]))
            except Exception:
                pass
    return sorted(l_vals)



In [ ]:
# 1) categories
for reg in regdict:
    soup = get_soup(regdict[reg])
    cat_as = soup.select("div.categories-list.no-margin-bottom a[href]")
    category_urls = []
    for a in cat_as:
        href = a.get("href")
        if href:
            category_urls.append(urljoin(START_URL, href))
    category_urls = list(dict.fromkeys(category_urls))

    visited_posts = set()

    # 2) for each category: collect all post links (paginate with ?l=1,2,...)
    for cat_url in category_urls:
        if 'managers-from-eea' in cat_url or 'service-providers-from-the-eea' in cat_url:
            continue

        base_cat = cat_url if cat_url.endswith("/") else cat_url + "/"

        first_soup = get_soup(cat_url)
        l_pages = extract_l_pages(first_soup, cat_url)
        sub_cat = first_soup.select("div.categories-list.no-margin-bottom a[href]")

        if l_pages:
            list_pages = [base_cat + f"?l={i}" for i in range(1, max(l_pages) + 1)]

        else:
            # Probe ?l=1.. until empty or repeats (covers sites that don't render all pagination links)
            list_pages = []
            prev_fp = None
            for i in range(1, 200):
                u = base_cat + f"?l={i}"
                s = get_soup(u)
                urls = extract_post_urls(s, u)
                fp = tuple(urls[:10])
                if not urls or fp == prev_fp:
                    break
                prev_fp = fp
                list_pages.append(u)

            # If probing produced nothing, fall back to original next-link pagination on cat_url
            if not list_pages:
                list_pages = [cat_url]

        for page_url in list_pages:
            soup = get_soup(page_url)
            post_urls = extract_post_urls(soup, page_url)
            for post_url in post_urls:
                if post_url in visited_posts:
                    continue
                visited_posts.add(post_url)

                dsoup = get_soup(post_url)
                h2 = dsoup.select_one("h2")
                title = (h2.get_text(" ", strip=True) if h2 else "").strip()

                pairs = {}
                for block in dsoup.select("div.row div.info-block"):
                    label_els = block.select(".market-item.label")
                    if label_els:
                        for lab in label_els:
                            label = norm_label(lab.get_text(" ", strip=True))
                            val_el = lab.find_next_sibling(
                                lambda t: t.name == "div"
                                and "market-item" in (t.get("class") or [])
                                and "label" not in (t.get("class") or [])
                            )
                            value = val_el.get_text(" ", strip=True) if val_el else ""
                            if label:
                                pairs[label] = value

                row = {k: "" for k in sqldict.keys()}
                row["Name"] = title
                row["Address_1"] = pairs.get("legal address", "") or pairs.get("address", "")
                row["Website"] = pairs.get("website", "") or pairs.get("web site", "") or pairs.get("web", "")
                row["Email"] = pairs.get("e-mail", "") or pairs.get("email", "")
                row["Phone"] = pairs.get("phone", "") or pairs.get("telephone", "") or pairs.get("tel.", "")
                row["Fax"] = pairs.get("fax", "")
                row["Check"] = post_url

                for k in sqldict.keys():
                    sqldict[k].append(row.get(k, ""))


In [10]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)

In [ ]:
from collections import deque
from urllib.parse import urlsplit, urlunsplit, urldefrag

def norm_url(u: str) -> str:
    u = urldefrag(u)[0]
    p = urlsplit(u)
    return urlunsplit((p.scheme, p.netloc, p.path.rstrip("/") + "/", "", ""))  # drop query/fragment

# 1) categories
for reg in regdict:
    soup = get_soup(regdict[reg])
    cat_as = soup.select("div.categories-list.no-margin-bottom a[href]")
    category_urls = []
    for a in cat_as:
        href = a.get("href")
        if href:
            category_urls.append(urljoin(START_URL, href))
    category_urls = list(dict.fromkeys(category_urls))

    visited_posts = set()

    # 2) for each category: dive subcategories + paginate (?l=1,2,...) and scrape posts
    for cat_url in category_urls:
        cat_queue = deque([norm_url(cat_url)])
        seen_cats = set()

        while cat_queue:
            cur_cat = norm_url(cat_queue.popleft())
            if cur_cat in seen_cats:
                continue
            if any(s in cur_cat for s in SKIP_SUBSTRINGS):
                continue
            seen_cats.add(cur_cat)

            first_soup = get_soup(cur_cat)

            # enqueue subcategories (but never enqueue ?l= pages)
            for a in first_soup.select("div.categories-list.no-margin-bottom a[href]"):
                href = a.get("href")
                if not href:
                    continue
                sub_url = norm_url(urljoin(cur_cat, href))
                if sub_url not in seen_cats:
                    cat_queue.append(sub_url)

            base_cur = cur_cat  # already normalized with trailing "/"

            # build listing pages for this category
            l_pages = extract_l_pages(first_soup, cur_cat)
            if l_pages:
                list_pages = [base_cur + f"?l={i}" for i in range(1, max(l_pages) + 1)]
            else:
                list_pages = []
                prev_fp = None
                for i in range(1, 200):
                    u = base_cur + f"?l={i}"
                    s = get_soup(u)
                    urls = extract_post_urls(s, u)
                    fp = tuple(urls[:10])
                    if not urls or fp == prev_fp:
                        break
                    prev_fp = fp
                    list_pages.append(u)
                if not list_pages:
                    list_pages = [cur_cat]

            # scrape posts from listing pages
            for page_url in list_pages:
                soup = first_soup if page_url == cur_cat else get_soup(page_url)
                post_urls = extract_post_urls(soup, page_url)

                for post_url in post_urls:
                    if post_url in visited_posts:
                        continue
                    visited_posts.add(post_url)

                    dsoup = get_soup(post_url)
                    h2 = dsoup.select_one("h2")
                    title = (h2.get_text(" ", strip=True) if h2 else "").strip()

                    pairs = {}
                    for block in dsoup.select("div.row div.info-block"):
                        for lab in block.select(".market-item.label"):
                            label = norm_label(lab.get_text(" ", strip=True))
                            val_el = lab.find_next_sibling(
                                lambda t: t.name == "div"
                                and "market-item" in (t.get("class") or [])
                                and "label" not in (t.get("class") or [])
                            )
                            value = val_el.get_text(" ", strip=True) if val_el else ""
                            if label:
                                pairs[label] = value

                    row = {k: "" for k in sqldict.keys()}
                    print(title)
                    row["Name"] = title
                    row["Address_1"] = pairs.get("legal address", "") or pairs.get("address", "")
                    row["Website"] = pairs.get("website", "") or pairs.get("web site", "") or pairs.get("web", "")
                    row["Email"] = pairs.get("e-mail", "") or pairs.get("email", "")
                    row["Phone"] = pairs.get("phone", "") or pairs.get("telephone", "") or pairs.get("tel.", "")
                    row["Fax"] = pairs.get("fax", "")
                    row["Check"] = post_url

                    for k in sqldict.keys():
                        sqldict[k].append(row.get(k, ""))


In [12]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)